# Task 6 — Full Text-to-Image Pipeline

**Goal:** Combine everything from Tasks 3, 4, and 5 into one end-to-end system:
**free-form text → CLIP text embedding → attention-GAN → generated flower image.**

**Key architectural change from Tasks 2 & 5:** those GANs conditioned on a
*class label* (an integer 0–101, via `nn.Embedding`) — which only works for
the 102 fixed flower classes. Task 6 needs to accept **arbitrary typed
sentences**, so we condition on a **CLIP text embedding** (a continuous
512-dim vector) instead. This is what makes it a genuine *text-to-image*
system rather than a label-to-image one.

**Reused components:**
- Task 3's CLIP tokenizer + text encoder (frozen — not retrained here)
- Task 4's caption-generation approach, applied per-image as training targets
- Task 5's self-attention Generator/Discriminator architecture, adapted to
  take a projected CLIP embedding instead of a label embedding

**Design choices (carried over):** dataset + captions built entirely on local
disk; Drive touched once at the end to save results.

## 1. Setup

In [ ]:
!pip install -q torch torchvision transformers matplotlib

import os
import shutil
import random
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import Flowers102
from torchvision.utils import make_grid, save_image
from transformers import CLIPTokenizer, CLIPTextModel
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 2. Load Oxford-102 and class names (same as Task 4)

In [ ]:
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)
IMG_SIZE = 64

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),
])

train_set = Flowers102(root=DATA_ROOT, split='train', download=True, transform=transform)
val_set   = Flowers102(root=DATA_ROOT, split='val', download=True, transform=transform)

CLASS_NAMES = [
    'pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea',
    'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood',
    'globe thistle', 'snapdragon', "colt's foot", 'king protea', 'spear thistle',
    'yellow iris', 'globe flower', 'purple coneflower', 'peruvian lily', 'balloon flower',
    'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary', 'red ginger',
    'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian',
    'artichoke', 'sweet william', 'carnation', 'garden phlox', 'love in the mist',
    'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower',
    'great masterwort', 'siam tulip', 'lenten rose', 'barbeton daisy', 'daffodil',
    'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold',
    'buttercup', 'oxeye daisy', 'common dandelion', 'petunia', 'wild pansy', 'primula',
    'sunflower', 'pelargonium', 'bishop of llandaff', 'gaura', 'geranium',
    'orange dahlia', 'pink-yellow dahlia', 'cautleya spicata', 'japanese anemone',
    'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum',
    'spring crocus', 'bearded iris', 'windflower', 'tree poppy', 'gazania',
    'azalea', 'water lily', 'rose', 'thorn apple', 'morning glory', 'passion flower',
    'lotus', 'toad lily', 'anthurium', 'frangipani', 'clematis', 'hibiscus',
    'columbine', 'desert-rose', 'tree mallow', 'magnolia', 'cyclamen', 'watercress',
    'canna lily', 'hippeastrum', 'bee balm', 'ball moss', 'foxglove', 'bougainvillea',
    'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower',
    'trumpet creeper', 'blackberry lily'
]

CAPTION_TEMPLATES = [
    'a photo of a {}',
    'a close-up photo of a {} flower',
    'a beautiful {} in bloom',
    'an image showing a {} flower',
    'a {} flower with vivid petals',
]

def make_caption(label_idx, seed=None):
    name = CLASS_NAMES[label_idx]
    rng = random.Random(seed)
    return rng.choice(CAPTION_TEMPLATES).format(name)

full_train = torch.utils.data.ConcatDataset([train_set, val_set])
print(f'Total training images: {len(full_train)}')

## 3. Load CLIP (frozen) for text embedding

Same model as Task 3. We keep it in `eval()` mode and never update its
weights — it's a fixed feature extractor here.

In [ ]:
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL_NAME)
clip_text_encoder = CLIPTextModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_text_encoder.eval()
for p in clip_text_encoder.parameters():
    p.requires_grad = False

CLIP_EMBED_DIM = clip_text_encoder.config.hidden_size  # 512 for this model
print('CLIP text embedding dimension:', CLIP_EMBED_DIM)

@torch.no_grad()
def encode_text(captions, max_length=32):
    inputs = tokenizer(captions, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(device)
    return clip_text_encoder(**inputs).pooler_output  # (batch, CLIP_EMBED_DIM)

# sanity check
test_emb = encode_text(['a photo of a rose'])
print('Sample embedding shape:', test_emb.shape)

## 4. Dataset wrapper: image + on-the-fly caption

Wraps the flower dataset so each `__getitem__` returns an (image, caption)
pair. We generate captions on the fly using the same templated approach as
Task 4, keyed by a stable seed so captions are reproducible across epochs.

In [ ]:
class CaptionedFlowers(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        caption = make_caption(label, seed=idx)
        return img, caption, label  # keep label too, just for later inspection/eval

captioned_dataset = CaptionedFlowers(full_train)

BATCH_SIZE = 64
dataloader = DataLoader(captioned_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

# quick check
imgs, caps, labels = next(iter(dataloader))
print('Batch image shape:', imgs.shape)
print('Sample captions:', caps[:3])

## 5. Generator and Discriminator conditioned on CLIP embeddings

Same self-attention architecture as Task 5, but the conditioning input
changes: instead of `nn.Embedding(num_classes, embed_dim)` looking up a
label, we project the continuous CLIP embedding down to a smaller dimension
via a learned linear layer. This is what enables generalizing to arbitrary
text rather than only the 102 fixed classes.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.key   = nn.Conv2d(in_channels, in_channels // 8, kernel_size=1)
        self.value = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        B, C, H, W = x.size()
        N = H * W
        q = self.query(x).view(B, -1, N).permute(0, 2, 1)
        k = self.key(x).view(B, -1, N)
        v = self.value(x).view(B, -1, N)
        attn = self.softmax(torch.bmm(q, k))
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return self.gamma * out + x, attn


TEXT_PROJ_DIM = 128   # projected dimension of the CLIP embedding used for conditioning
LATENT_DIM = 100

class TextConditionedGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, clip_dim=CLIP_EMBED_DIM, text_proj_dim=TEXT_PROJ_DIM, img_size=IMG_SIZE):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(clip_dim, text_proj_dim),
            nn.LeakyReLU(0.2, inplace=True),
        )
        input_dim = latent_dim + text_proj_dim
        self.init_size = img_size // 16
        self.fc = nn.Linear(input_dim, 256 * self.init_size * self.init_size)

        self.block1 = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(256, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.attn = SelfAttention(64)
        self.block3 = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(64, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(32, 3, 3, stride=1, padding=1),
            nn.Tanh(),
        )

    def forward(self, noise, text_embed):
        text_feat = self.text_proj(text_embed)
        x = torch.cat([noise, text_feat], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), 256, self.init_size, self.init_size)
        x = self.block1(x)
        x = self.block2(x)
        x, _ = self.attn(x)
        return self.block3(x)


class TextConditionedDiscriminator(nn.Module):
    def __init__(self, clip_dim=CLIP_EMBED_DIM, text_proj_dim=TEXT_PROJ_DIM, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.text_proj = nn.Sequential(
            nn.Linear(clip_dim, img_size * img_size),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(4, 32, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.attn = SelfAttention(64)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.adv_layer = nn.Sequential(
            nn.Linear(256 * (img_size // 16) ** 2, 1),
            nn.Sigmoid()
        )

    def forward(self, img, text_embed):
        text_map = self.text_proj(text_embed).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, text_map], dim=1)
        x = self.block1(x)
        x = self.block2(x)
        x, _ = self.attn(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        return self.adv_layer(x)


generator = TextConditionedGenerator().to(device)
discriminator = TextConditionedDiscriminator().to(device)
print('Generator and Discriminator built.')

## 6. Training setup

In [ ]:
adversarial_loss = nn.BCELoss()
lr = 0.0002
beta1 = 0.5
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

N_EPOCHS = 100
SAMPLE_EVERY = 10

os.makedirs('/content/outputs', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

# Fixed prompts to track across training — real free-form text, not just class names
FIXED_PROMPTS = [
    'a photo of a red rose',
    'a yellow sunflower in bloom',
    'a purple orchid flower',
    'a white daisy with a yellow center',
    'a pink lotus flower',
    'a bright orange marigold',
]
fixed_text_embeds = encode_text(FIXED_PROMPTS)
fixed_noise = torch.randn(len(FIXED_PROMPTS), LATENT_DIM, device=device)

## 7. Training loop

Same adversarial procedure as Tasks 2/5, but the conditioning signal comes
from live CLIP-encoded captions each batch (encoded with `torch.no_grad()`
since CLIP stays frozen).

**Time estimate:** similar to Task 5, plus a small overhead for CLIP encoding
each batch (CLIP inference is cheap relative to GAN training). Budget
~40–60 minutes for 100 epochs on a T4.

In [ ]:
g_losses, d_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0

    for real_imgs, captions, _ in dataloader:
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        text_embeds = encode_text(list(captions))  # frozen CLIP, no grad tracked into it

        valid = torch.ones(bs, 1, device=device)
        fake = torch.zeros(bs, 1, device=device)

        # ---- Train Generator ----
        optimizer_G.zero_grad()
        noise = torch.randn(bs, LATENT_DIM, device=device)
        gen_imgs = generator(noise, text_embeds)
        g_loss = adversarial_loss(discriminator(gen_imgs, text_embeds), valid)
        g_loss.backward()
        optimizer_G.step()

        # ---- Train Discriminator ----
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs, text_embeds), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach(), text_embeds), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()

    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    g_losses.append(avg_g)
    d_losses.append(avg_d)
    print(f'Epoch [{epoch}/{N_EPOCHS}]  D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}')

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        generator.eval()
        with torch.no_grad():
            samples = generator(fixed_noise, fixed_text_embeds)
        grid = make_grid(samples, nrow=len(FIXED_PROMPTS), normalize=True)
        save_image(grid, f'/content/outputs/epoch_{epoch:03d}.png')
        generator.train()

torch.save(generator.state_dict(), '/content/checkpoints/text2img_generator.pth')
torch.save(discriminator.state_dict(), '/content/checkpoints/text2img_discriminator.pth')
print('\nTraining complete.')

## 8. Plot training curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(g_losses, label='Generator loss')
plt.plot(d_losses, label='Discriminator loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Text-to-Image Pipeline Training Losses')
plt.legend()
plt.tight_layout()
plt.savefig('/content/outputs/training_curves.png', dpi=150)
plt.show()

## 9. The full pipeline as one function

This is the actual deliverable: a single function taking a raw text string
and returning a generated image — text preprocessing, embedding, and GAN
generation all chained together.

In [ ]:
def text_to_image(prompt: str, n_samples: int = 4, seed: int = None):
    """Full pipeline: raw text -> CLIP embedding -> GAN-generated image(s)."""
    generator.eval()
    if seed is not None:
        torch.manual_seed(seed)
    with torch.no_grad():
        text_embed = encode_text([prompt])                       # (1, CLIP_EMBED_DIM)
        text_embed = text_embed.repeat(n_samples, 1)              # same prompt, several noise seeds
        noise = torch.randn(n_samples, LATENT_DIM, device=device)
        imgs = generator(noise, text_embed)
    grid = make_grid(imgs, nrow=n_samples, normalize=True)
    plt.figure(figsize=(3 * n_samples, 3))
    plt.imshow(grid.permute(1, 2, 0).cpu())
    plt.axis('off')
    plt.title(f'"{prompt}"')
    plt.show()
    generator.train()

# Try it out on prompts never explicitly seen as exact captions during training
text_to_image('a bright red rose', seed=1)
text_to_image('a yellow sunflower', seed=1)
text_to_image('a delicate white orchid', seed=1)

## 10. Copy results to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/elevance-skills/task6_full_pipeline'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree('/content/outputs', os.path.join(DRIVE_DIR, 'outputs'), dirs_exist_ok=True)
shutil.copytree('/content/checkpoints', os.path.join(DRIVE_DIR, 'checkpoints'), dirs_exist_ok=True)

print('Copied outputs and checkpoints to:', DRIVE_DIR)

## 11. Summary of findings

Fill this in after training, then copy into `NOTES.md` and today's daily log:
- Did the pipeline respond sensibly to prompts describing colors/flower types
  it saw variations of during training?
- How did it handle a prompt that combines concepts in a new way (e.g. a
  color not strongly associated with a given flower in training data)?
- How does overall image quality compare to Task 5 (label-conditioned)?
- What are the clear next steps to make this more robust (more training data,
  longer training, larger images, larger GAN)?